In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except ImportError:
    pass


def logistic_step(x, r):
    """One update; use x in [0, 1] and r in [0, 4]."""
    return r * x * (1 - x)


def cobweb_points_up_to(r, frame, x0=0.1):
    """Return path, last completed state, and completed iterations.

    Frame 0 is (x0, 0). Each iteration has two movements:
    vertical to the curve, then horizontal to the diagonal.
    """
    if not 0 <= r <= 4 or not 0 <= x0 <= 1:
        raise ValueError("Use r in [0, 4] and x0 in [0, 1].")
    if not isinstance(frame, (int, np.integer)) or frame < 0:
        raise ValueError("frame must be a nonnegative integer.")
    x = float(x0)
    points = [(x, 0.0)]
    for _ in range(frame // 2):
        next_x = logistic_step(x, r)
        points.extend([(x, next_x), (next_x, next_x)])
        x = next_x
    if frame % 2:
        points.append((x, logistic_step(x, r)))
    return np.asarray(points), x, frame // 2


def estimate_period(r, x0=0.1, max_period=16, burn_in=12000,
                    block_size=512, tolerance=1e-10):
    """Numerical evidence for a stable cycle on this trajectory, not proof.

    Require ordered recurrence in two consecutive blocks and an attracting
    cycle multiplier. None means unresolved within these settings, not chaos.
    """
    if not 0 <= r <= 4 or not 0 <= x0 <= 1:
        raise ValueError("Use r in [0, 4] and x0 in [0, 1].")
    if any(not isinstance(v, (int, np.integer)) for v in
           (max_period, burn_in, block_size)):
        raise ValueError("Iteration and period settings must be integers.")
    if max_period < 1 or burn_in < 0 or block_size < 8 * max_period:
        raise ValueError("Use max_period >= 1, burn_in >= 0, block_size >= 8*max_period.")
    if not np.isfinite(tolerance) or tolerance <= 0:
        raise ValueError("tolerance must be positive and finite.")
    x = float(x0)
    for _ in range(burn_in):
        x = logistic_step(x, r)
    values = np.empty(2 * block_size)
    for i in range(len(values)):
        x = logistic_step(x, r)
        values[i] = x
    # Smallest recurring lag preserves temporal order; no rounding or bins.
    for period in range(1, max_period + 1):
        if np.max(np.abs(values[period:] - values[:-period])) > tolerance:
            continue
        stable = True
        for end in (block_size, 2 * block_size):
            cycle = values[end - period:end]
            multiplier = abs(np.prod(r * (1 - 2 * cycle)))
            if multiplier >= 1 - 1e-6:
                stable = False
                break
        if stable:
            return period
    return None


def make_cobweb_demo(x0=0.1, max_iterations=40):
    """Create independent controls; changing r starts a fresh experiment."""
    play = widgets.Play(value=0, min=0, max=2 * max_iterations,
                        step=1, interval=250, repeat=False)
    frame = widgets.IntSlider(value=0, min=0, max=play.max, step=1,
                              description="Movement:", continuous_update=False)
    r_slider = widgets.FloatSlider(value=2.5, min=2.5, max=3.99, step=0.01,
                                   description="r:", readout_format=".2f",
                                   continuous_update=False,
                                   layout=widgets.Layout(width="420px"))
    reset = widgets.Button(description="Reset")
    output = widgets.Output()
    # A kernel link keeps the frame observer synchronized with Play.
    link = widgets.link((play, "value"), (frame, "value"))
    period_cache = {}

    def draw(change=None):
        r = float(r_slider.value)
        movement = int(frame.value)
        points, _, iteration = cobweb_points_up_to(r, movement, x0)
        if r not in period_cache:
            period_cache[r] = estimate_period(r, x0=x0)
        period = period_cache[r]
        label = (f"Estimated attractor period: {period}" if period is not None
                 else "No stable period <= 16 confirmed")
        phase = ("Initial condition" if movement == 0 else
                 "Apply the rule" if movement % 2 else "Output becomes input")
        with output:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(6, 6))
            xs = np.linspace(0, 1, 400)
            ax.plot(xs, logistic_step(xs, r), "k", label="The rule")
            ax.plot(xs, xs, "k--", alpha=0.4, label="$x_{t+1}=x_t$")
            ax.plot(points[:, 0], points[:, 1], color="tab:red", linewidth=1)
            ax.scatter(*points[-1], color="tab:red", s=45, zorder=5)
            ax.set(xlim=(0, 1), ylim=(0, 1), xlabel="$x_t$", ylabel="$x_{t+1}$",
                   title=f"r = {r:.2f} | Completed iterations: {iteration}\n{label}")
            ax.set_aspect("equal", adjustable="box")
            ax.legend(loc="upper left")
            fig.text(0.5, 0.02, phase, ha="center")
            fig.tight_layout(rect=(0, 0.05, 1, 1))
            plt.show()
            plt.close(fig)

    def restart(change=None):
        play.playing = False
        if frame.value != 0:
            frame.value = 0  # The observer draws exactly once.
        else:
            draw()

    frame.observe(draw, names="value")
    r_slider.observe(restart, names="value")
    reset.on_click(restart)
    panel = widgets.VBox([
        r_slider, widgets.HBox([play, reset]), frame,
        widgets.HTML("Period estimated separately after 12,000 settling updates; "
                     "the red path includes the transient. An unconfirmed period "
                     "does not establish chaos."), output,
    ])
    display(panel)
    draw()
    return panel, link


cobweb_demo, cobweb_link = make_cobweb_demo()
